# Adding Sunglasses for a Cool New Look

**Name :** MUTHU MANIKKAM G    **Reg. No :** 212225100029

**Aim:** Detect the face in a photo using OpenCV and overlay a sunglasses image on the eyes.

**Files needed in the same folder as this notebook:**
- `face.jpg` – the portrait photo
- `sunglass.jpg` – the sunglasses image (white background)

## 1. Import libraries

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## 2. Load the images

In [ ]:
face_img = cv2.imread("face.jpg")        # BGR image
glass_img = cv2.imread("sunglass.jpg")   # BGR image with white background

print("Face image shape     :", face_img.shape)
print("Sunglasses image shape:", glass_img.shape)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB));  ax[0].set_title("Original photo");   ax[0].axis("off")
ax[1].imshow(cv2.cvtColor(glass_img, cv2.COLOR_BGR2RGB)); ax[1].set_title("Sunglasses image"); ax[1].axis("off")
plt.show()

## 3. Remove the white background of the sunglasses
The sunglasses image is a JPG, so it has no transparency. We flood-fill the white background starting
from the image borders (so white parts *inside* the frame are kept) and turn it into a transparent **alpha** channel.

In [ ]:
def make_transparent(bgr, tol=12):
    h, w = bgr.shape[:2]
    # Pad with a white border so the flood fill can reach the whole outer background
    padded = cv2.copyMakeBorder(bgr, 2, 2, 2, 2, cv2.BORDER_CONSTANT, value=(255, 255, 255))
    ff_mask = np.zeros((padded.shape[0] + 2, padded.shape[1] + 2), np.uint8)
    cv2.floodFill(padded, ff_mask, (0, 0), (0, 0, 0),
                  loDiff=(tol,) * 3, upDiff=(tol,) * 3,
                  flags=4 | cv2.FLOODFILL_MASK_ONLY | (255 << 8))
    background = ff_mask[1:-1, 1:-1][2:-2, 2:-2]          # 255 where background
    alpha = np.where(background > 0, 0, 255).astype(np.uint8)
    alpha = cv2.GaussianBlur(alpha, (3, 3), 0)             # soften the edges a little
    return np.dstack([bgr, alpha])                         # BGRA

glass_rgba = make_transparent(glass_img)

# Crop the empty margin so the sunglasses fill the image exactly
ys, xs = np.where(glass_rgba[:, :, 3] > 10)
glass_rgba = glass_rgba[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
cv2.imwrite("sunglass.png", glass_rgba)
print("Transparent sunglasses shape:", glass_rgba.shape)

# Preview on a grey background to confirm the transparency works
bg = np.full(glass_rgba.shape[:2] + (3,), 128, np.uint8)
a = glass_rgba[:, :, 3:4] / 255.0
preview = (glass_rgba[:, :, :3] * a + bg * (1 - a)).astype(np.uint8)
plt.figure(figsize=(6, 3)); plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.title("Background removed"); plt.show()

## 4. Detect the face and eyes
We use OpenCV's pre-trained **Haar cascade** classifiers. The face is found first, then the eyes are searched for
only in the upper half of the face.

In [ ]:
gray = cv2.cvtColor(face_img, cv2.COLOR_BGR2GRAY)

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
eye_cascade  = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_eye.xml")

faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
print("Faces found:", len(faces))

# Keep the largest face
x, y, w, h = max(faces, key=lambda f: f[2] * f[3])

# Look for eyes in the top part of the face only
roi_gray = gray[y:y + int(h * 0.6), x:x + w]
eyes = eye_cascade.detectMultiScale(roi_gray, scaleFactor=1.1, minNeighbors=6, minSize=(w // 10, w // 10))
print("Eyes found:", len(eyes))

boxed = face_img.copy()
cv2.rectangle(boxed, (x, y), (x + w, y + h), (0, 255, 0), 2)
for (ex, ey, ew, eh) in eyes:
    cv2.rectangle(boxed, (x + ex, y + ey), (x + ex + ew, y + ey + eh), (255, 0, 0), 2)

plt.figure(figsize=(4, 6)); plt.imshow(cv2.cvtColor(boxed, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.title("Detected face & eyes"); plt.show()

## 5. Decide where the sunglasses go
- If **two valid eyes** are detected, the sunglasses are centred between them and sized from the distance between the eyes.
- Otherwise we fall back to face proportions: eyes sit about **40 %** down the face box and the glasses are about as wide as the face.

In [ ]:
# Discard false detections: eyes must be side by side, not stacked
good = len(eyes) >= 2
if good:
    best = sorted(eyes, key=lambda e: e[2] * e[3], reverse=True)[:2]
    best = sorted(best, key=lambda e: e[0])
    dx = abs((best[1][0] + best[1][2] // 2) - (best[0][0] + best[0][2] // 2))
    dy = abs((best[1][1] + best[1][3] // 2) - (best[0][1] + best[0][3] // 2))
    good = dx > 0.25 * w and dy < 0.5 * dx

if good:
    # take the two biggest detections, ordered left -> right
    best = sorted(eyes, key=lambda e: e[2] * e[3], reverse=True)[:2]
    best = sorted(best, key=lambda e: e[0])
    centres = [(x + ex + ew // 2, y + ey + eh // 2) for ex, ey, ew, eh in best]
    cx = (centres[0][0] + centres[1][0]) // 2
    cy = (centres[0][1] + centres[1][1]) // 2
    eye_dist = abs(centres[1][0] - centres[0][0])
    glass_w = int(max(eye_dist * 2.6, w * 0.95))   # never smaller than ~face width
    print("Placement based on detected eyes")
else:
    cx = x + w // 2
    cy = y + int(h * 0.40)
    glass_w = int(w * 1.0)
    print("Placement based on face proportions")

glass_h = int(glass_w * glass_rgba.shape[0] / glass_rgba.shape[1])   # keep aspect ratio
print("Centre:", (cx, cy), " Sunglasses size:", (glass_w, glass_h))

## 6. Overlay the sunglasses (alpha blending)

In [ ]:
def overlay(background, fg_rgba, center, size):
    out = background.copy()
    fg = cv2.resize(fg_rgba, size, interpolation=cv2.INTER_AREA)
    fw, fh = size
    x0, y0 = center[0] - fw // 2, center[1] - fh // 2

    # Clip if the sunglasses go outside the photo
    bx0, by0 = max(x0, 0), max(y0, 0)
    bx1, by1 = min(x0 + fw, out.shape[1]), min(y0 + fh, out.shape[0])
    fg = fg[by0 - y0:by1 - y0, bx0 - x0:bx1 - x0]

    alpha = fg[:, :, 3:4] / 255.0
    roi = out[by0:by1, bx0:bx1].astype(float)
    out[by0:by1, bx0:bx1] = (fg[:, :, :3] * alpha + roi * (1 - alpha)).astype(np.uint8)
    return out

result = overlay(face_img, glass_rgba, (cx, cy), (glass_w, glass_h))

fig, ax = plt.subplots(1, 2, figsize=(9, 6))
ax[0].imshow(cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)); ax[0].set_title("Before"); ax[0].axis("off")
ax[1].imshow(cv2.cvtColor(result,   cv2.COLOR_BGR2RGB)); ax[1].set_title("After - cool new look!"); ax[1].axis("off")
plt.show()

## 7. Save the output

In [ ]:
cv2.imwrite("face_with_sunglasses.jpg", result)
print("Saved as face_with_sunglasses.jpg")

## Result
The face was detected with a Haar cascade, the sunglasses' white background was converted to a transparent alpha channel,
and the glasses were resized and alpha-blended over the eyes to give the photo a new look.